In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import os
import glob
import re
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec

# 设置绘图样式
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})
# ================= Set Paths =================
RESULTS_ROOT = "./pinn_results"

# Find latest PINN run directory
pinn_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_pi100_K50_*")))
if not pinn_runs:
    raise FileNotFoundError("No PINN results directory found")
pinn_dir = pinn_runs[-1]  # take the latest run

# Find latest Monte Carlo run directory
mc_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_mc_pi100_K50_*")))
if not mc_runs:
    raise FileNotFoundError("No Monte Carlo results directory found")
mc_dir = mc_runs[-1]  # take the latest run

# Create comparison results directory
comparison_dir = os.path.join("generated", "comparison_results")
os.makedirs(comparison_dir, exist_ok=True)
print(f"Comparison results will be saved to: {comparison_dir}")

# Load parameters from PINN run (assume parameters are similar for both)
try:
    with open(os.path.join(pinn_dir, "simulation_parameters.txt"), "r") as f:
        params_text = f.read()
        # Extract key parameters
        tau = float(re.search(r"Tau = ([\d.]+)", params_text).group(1))
        K = float(re.search(r"Strike price \(K\) = ([\d.]+)", params_text).group(1))
        pi = float(re.search(r"Pi = ([\d.]+)", params_text).group(1))
except:
    # Default values if file not found
    tau = 3.0
    K = 50.0
    pi = 100.0

# ================= Load Data =================
# Load PINN results
pinn_data = np.load(os.path.join(pinn_dir, "pinn_results.npy"))
A_pinn = pinn_data[:, 0]
t_pinn = pinn_data[:, 1]
C_pinn = pinn_data[:, 2]

# Load Monte Carlo results
mc_data = np.load(os.path.join(mc_dir, "mc_results.npy"))
A_mc = mc_data[:, 0]
t_mc = mc_data[:, 1]
C_mc = mc_data[:, 2]

# Print basic info
print(f"PINN data points: {len(A_pinn)}")
print(f"Monte Carlo data points: {len(A_mc)}")
print(f"PINN price range: {C_pinn.min():.2f} - {C_pinn.max():.2f}")
print(f"Monte Carlo price range: {C_mc.min():.2f} - {C_mc.max():.2f}")

# ================= Data Alignment =================
# Create common grid
A_min = max(min(A_pinn), min(A_mc))
A_max = min(max(A_pinn), max(A_mc))
t_min = max(min(t_pinn), min(t_mc))
t_max = min(max(t_pinn), max(t_mc))

print(f"Common region: A={A_min:.1f}-{A_max:.1f}, t={t_min:.1f}-{t_max:.1f}")

# Create interpolation grid
grid_points = 100
A_grid = np.linspace(A_min, A_max, grid_points)
t_grid = np.linspace(t_min, t_max, grid_points)
A_mesh, t_mesh = np.meshgrid(A_grid, t_grid, indexing='ij')

# Interpolate PINN data to grid
C_pinn_grid = griddata(
    (A_pinn, t_pinn), 
    C_pinn, 
    (A_mesh, t_mesh), 
    method='cubic'
)

# Interpolate Monte Carlo data to grid
C_mc_grid = griddata(
    (A_mc, t_mc), 
    C_mc, 
    (A_mesh, t_mesh), 
    method='cubic'
)

# Calculate errors
abs_error = C_pinn_grid - C_mc_grid
rel_error = abs_error / (np.abs(C_mc_grid) + 1e-6) * 100  # percentage error

# For plotting, create flattened data
A_flat = A_mesh.ravel()
t_flat = t_mesh.ravel()
C_pinn_flat = C_pinn_grid.ravel()
C_mc_flat = C_mc_grid.ravel()
abs_error_flat = abs_error.ravel()
rel_error_flat = rel_error.ravel()


# ================= Integrated Comparison Figure =================
print("Generating integrated comparison figure...")

# Create symmetric colormap for errors
# error_norm = colors.TwoSlopeNorm(vmin=-np.max(np.abs(abs_error)), vcenter=0, vmax=np.max(np.abs(abs_error)))

fig = plt.figure(figsize=(28, 12))
# fig.suptitle('Integrated Carbon Option Price Analysis', fontsize=18, y=0.98)

# Create GridSpec: 2 rows (top: 3 plots, bottom: 5 plots)
gs = gridspec.GridSpec(2, 1, height_ratios=[1.5, 1])
gs_top = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[0])
gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=gs[1])

# ===== Top Row: 2D Contour Views =====
# PINN results
ax1 = plt.subplot(gs_top[0])
contour1 = ax5.contourf(
    A_mesh, t_mesh, C_pinn_grid, 
    levels=50, 
    cmap='viridis'
)
ax5.set_xlabel('Asset Price (A)')
ax1.set_ylabel('Time (t)')
ax1.set_title('PINN Price Surface')
plt.colorbar(contour1, ax=ax1, label='Option Price (C)')

# Monte Carlo results
ax2 = plt.subplot(gs_top[1])
contour2 = ax2.contourf(
    A_mesh, t_mesh, C_mc_grid, 
    levels=50, 
    cmap='viridis'
)
ax2.set_xlabel('Asset Price (A)')
ax2.set_ylabel('Time (t)')
ax2.set_title('Monte Carlo Price Surface')
plt.colorbar(contour2, ax=ax2, label='Option Price (C)')

# Relative error
ax3 = plt.subplot(gs_top[2])
contour3 = ax3.contourf(
    A_mesh, t_mesh, np.abs(rel_error), 
    levels=50, 
    cmap='hot'
)
ax3.set_xlabel('Asset Price (A)')
ax3.set_ylabel('Time (t)')
ax3.set_title('Absolute Relative Error (%)')
cbar3 = plt.colorbar(contour3, ax=ax3, label='Relative Error (%)')

# ===== Bottom Row: Time Series Comparisons =====
# Select first 5 time points
time_points = time_points[:5] if len(time_points) >= 5 else time_points
# 创建一个空的图例句柄和标签列表
handles = []
labels = []

for i, t_val in enumerate(time_points):
    ax = plt.subplot(gs_bottom[i])
    
    # Find closest time index
    t_idx = np.argmin(np.abs(t_grid - t_val))
    
    # Extract results
    A_at_t = A_grid
    pinn_at_t = C_pinn_grid[:, t_idx]
    mc_at_t = C_mc_grid[:, t_idx]
    
    # Plot both methods
    # 绘制两种方法的结果
    line1, = ax.plot(A_at_t, pinn_at_t, 'b-', linewidth=2.5, label='PINN')
    line2, = ax.plot(A_at_t, mc_at_t, 'r--', linewidth=2, label='Monte Carlo')
    
    # 绘制行权价格
    line3 = ax.axvline(x=K, color='g', linestyle=':', alpha=0.7, label='Strike Price')
    
    # 如果是第一次循环，保存图例句柄
    if i == 0:
        handles = [line1, line2, line3]
        labels = [h.get_label() for h in handles]
    # Plot strike price
    ax.axvline(x=K, color='g', linestyle=':', alpha=0.7, label='Strike Price')
    
    # Calculate and display MAE
    mae = np.mean(np.abs(abs_error[:, t_idx]))
    ax.text(0.05, 0.95, f"MAE: {mae:.4f}", 
            transform=ax.transAxes, 
            fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Asset Price (A)')
    ax.set_ylabel('Option Price (C)')
    ax.set_title(f't = {t_val:.2f}')
    ax.grid(True, alpha=0.3)
    
    # Only add legend to first subplot
    # if i == 0:
    #     ax.legend(loc='upper left')
# 在整个图形的底部中间添加图例
fig.legend(handles, labels, 
           loc='lower center', 
           bbox_to_anchor=(0.5, 0.02), 
           ncol=3,  # 一行显示三个图例项
           frameon=True, 
           fancybox=True, 
           shadow=True
           )
# Adjust layout and save
plt.tight_layout(rect=[0, 0.07, 1, 0.96])
integrated_fig_path = os.path.join(comparison_dir, "integrated_comparison.png")
plt.savefig(integrated_fig_path, dpi=300, bbox_inches='tight')
plt.show()

# 计算平均绝对相对误差
# MARE = np.mean(rel_error)
# print(f"全局平均绝对相对误差 (MARE): {MARE:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import os
import glob
import re
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import torch
import torch.nn as nn

# 设置绘图样式
plt.rcParams.update({
    'font.size': 24,  
    'font.weight': 'bold', 
    'axes.titlesize': 32, 
    'axes.labelsize': 28,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 28,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})


# ================= Set Paths and Load Data =================
RESULTS_ROOT = "./pinn_results"

# Find latest PINN run directory
pinn_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_pi100_K50_*")))
if not pinn_runs:
    raise FileNotFoundError("No PINN results directory found")
pinn_dir = pinn_runs[-1]
print(f"Loading results from: {pinn_dir}")
# Find latest Monte Carlo run directory
mc_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_mc_pi100_K50_*")))
if not mc_runs:
    raise FileNotFoundError("No Monte Carlo results directory found")
mc_dir = mc_runs[-1]

# Create comparison results directory
comparison_dir = os.path.join("generated", "comparison_results")
os.makedirs(comparison_dir, exist_ok=True)
SAVE_DIR = "./generated/one-period-result-fig"
# Load parameters
try:
    with open(os.path.join(pinn_dir, "simulation_parameters.txt"), "r") as f:
        params_text = f.read()
        tau = float(re.search(r"Tau = ([\d.]+)", params_text).group(1))
        K = float(re.search(r"Strike price \(K\) = ([\d.]+)", params_text).group(1))
        pi = float(re.search(r"Pi = ([\d.]+)", params_text).group(1))
except:
    tau = 3.0
    K = 50.0
    pi = 100.0

# ================= 重新创建模型用于分析 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class CarbonOptionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, A, t):
        if A.dim() == 1:
            A = A.view(-1, 1)
        if t.dim() == 1:
            t = t.view(-1, 1)
        x = torch.cat([A, t], dim=1)
        return self.net(x)

# 加载模型
model = CarbonOptionNN().to(device)
model_path = os.path.join(pinn_dir, "carbon_option_model.pth")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model loaded successfully")



time_points = np.arange(0, 3 + 0.1, 0.5)
colors = cm.viridis(np.linspace(0, 1, len(time_points)))
# Load PINN and Monte Carlo data
pinn_data = np.load(os.path.join(pinn_dir, "pinn_results.npy"))
A_pinn = pinn_data[:, 0]
t_pinn = pinn_data[:, 1]
C_pinn = pinn_data[:, 2]

mc_data = np.load(os.path.join(mc_dir, "mc_results.npy"))
A_mc = mc_data[:, 0]
t_mc = mc_data[:, 1]
C_mc = mc_data[:, 2]

# ================= Data Processing =================
# Create common grid
A_min = max(min(A_pinn), min(A_mc))
A_max = min(max(A_pinn), max(A_mc))
t_min = max(min(t_pinn), min(t_mc))
t_max = min(max(t_pinn), max(t_mc))

grid_points = 100
A_grid = np.linspace(A_min, A_max, grid_points)
t_grid = np.linspace(t_min, t_max, grid_points)
A_mesh, t_mesh = np.meshgrid(A_grid, t_grid, indexing='ij')

# Interpolate data
C_pinn_grid = griddata((A_pinn, t_pinn), C_pinn, (A_mesh, t_mesh), method='cubic')
C_mc_grid = griddata((A_mc, t_mc), C_mc, (A_mesh, t_mesh), method='cubic')

# Calculate errors
abs_error = C_pinn_grid - C_mc_grid
rel_error = abs_error / (np.abs(C_mc_grid) + 1e-6) * 100

# ================= Create Three-Row Layout Figure =================
print("Creating three-row layout figure...")

fig = plt.figure(figsize=(40, 16))

# Create 3x2 GridSpec (3 rows, 2 columns)
# gs = gridspec.GridSpec(2, 4, height_ratios=[1.2, 1])
# Create GridSpec: 2 rows (top: 3 plots, bottom: 5 plots)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1.2])
gs_top = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=gs[0])
gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1], width_ratios=[0.8, 1, 1])  
# ===== ROW 1: PINN Results =====
# Column 1: 3D Surface

ax4 = plt.subplot(gs_top[3])
# Select three representative time points



line_styles = ['-', '--', '-.']
# 创建图例句柄和标签
legend_handles = []
legend_labels = []

for i, t_val in enumerate(time_points):
    t_idx = np.argmin(np.abs(t_grid - t_val))
    
    # Plot both PINN and Monte Carlo
    ax4.plot(A_grid, C_pinn_grid[:, t_idx], 
             color=colors[i], 
             linestyle=line_styles[0],
             linewidth=2,
             label=f'PINN t={t_val:.2f}')
    
    ax4.plot(A_grid, C_mc_grid[:, t_idx], 
             color=colors[i], 
             linestyle=line_styles[1],
             linewidth=2,
             alpha=0.7,
             label=f'MC t={t_val:.2f}')

ax4.set_xlabel('Asset Price (A)')
ax4.set_ylabel('Option Price (C)')
ax4.set_title('(d) PINN vs MC at Time Points')
# ax5.legend()
ax4.grid(True, alpha=0.3)


# 创建全局图例句柄
global_time_handles = [mlines.Line2D([0], [0], color=colors[i], lw=4) 
                      for i in range(len(time_points))]
global_time_labels = [f't={t_val:.1f}' for t_val in time_points]

# 添加方法图例句柄
method_handles = [
    mlines.Line2D([0], [0], color='black', linestyle='-', lw=3, label='PINN'),
    mlines.Line2D([0], [0], color='black', linestyle='--', lw=3, label='MC')
]

# 合并所有图例句柄和标签
all_handles = method_handles + global_time_handles
all_labels = ['PINN', 'MC'] + global_time_labels
# 为ax4创建图例（放在右边竖排）
ax4_legend = ax4.legend(all_handles, all_labels, 
                       loc='center left',
                       bbox_to_anchor=(1.02, 0.5),
                       ncol=1,
                       frameon=True,
                       framealpha=0.9,
                       fontsize=20)

# 添加全局图例（在整个图形底部中央）
# fig.legend(all_handles, all_labels, 
#            loc='lower center', 
#            bbox_to_anchor=(0.5, 0.0),
#            ncol=len(time_points) + 2,  # 时间点数量 + 2个方法
#            frameon=True, 
#            framealpha=0.9)

# Column 2: 2D Contour
ax1 = plt.subplot(gs_top[0])
contour1 = ax1.contourf(A_mesh, t_mesh, C_pinn_grid, levels=50, cmap='viridis')
ax1.set_xlabel('Asset Price (A)')
ax1.set_ylabel('Time (t)')
ax1.set_title('(a) PINN Price 2D Surface')
plt.colorbar(contour1, ax=ax1, label='Option Price (C)')

# Column 3: Monte Carlo Price
ax2 = plt.subplot(gs_top[1])
contour2 = ax2.contourf(A_mesh, t_mesh, C_mc_grid, levels=50, cmap='viridis')
ax2.set_xlabel('Asset Price (A)')
ax2.set_ylabel('Time (t)')
ax2.set_title('(b) MC Price Surface')
plt.colorbar(contour2, ax=ax2, label='Option Price (C)')

# Column 4: Absolute Relative Error
ax3 = plt.subplot(gs_top[2])

rel_error_clipped = np.clip(np.abs(rel_error), 0, 100)
# contour4 = ax3.contourf(A_mesh, t_mesh, np.abs(rel_error), levels=50, cmap='hot')
contour3 = ax3.contourf(A_mesh, t_mesh, rel_error_clipped, levels=50, cmap='hot')
contour4 = ax3.contourf(A_mesh, t_mesh, np.abs(rel_error), levels=50, cmap='hot')

ax3.set_xlabel('Asset Price (A)')
ax3.set_ylabel('Time (t)')
ax3.set_title('(c) Absolute Relative Error')
# cbar = plt.colorbar(contour4, ax=ax4, label='Error (%)')


cbar = plt.colorbar(contour3, ax=ax3, label='Relative Error (%)')
# cbar.set_ticks(np.arange(0, 1, 0.1))  # 设置colorbar刻度
# cbar.formatter.set_powerlimits((0, 0))
# cbar.update_ticks()
# ===== ROW 2: Detailed Comparisons =====
# # Column 1: Price Curves at Different Times
# ax5 = fig.add_subplot(gs[1, 0])


# for i, t_val in enumerate(time_points):
#     t_idx = np.argmin(np.abs(t_grid - t_val))
#     ax5.plot(A_grid, C_pinn_grid[:, t_idx], 
#              label=f't = {t_val:.2f}', 
#              linewidth=2.5, 
#              color=colors[i])

# ax5.set_xlabel('Asset Price (A)')
# ax5.set_ylabel('Option Price (C)')
# ax5.set_title('Price Curves at Different Times')
# ax5.legend()
# ax5.grid(True, alpha=0.3)

# Column 2: Three Time Point Comparisons
ax5 = fig.add_subplot(gs_bottom[0], projection='3d')
surf1 = ax5.plot_surface(A_mesh, t_mesh, C_pinn_grid, cmap='viridis', alpha=0.9)
ax5.set_xlabel('Asset Price (A)', labelpad=12)
ax5.set_ylabel('Time (t)', labelpad=12)
ax5.set_zlabel('Option Price (C)', labelpad=12)

ax5.set_box_aspect([1, 1, 0.8])
ax5.set_title('(e) PINN 3D Solution Surface', pad=15)
# 重新定义计算PDE残差的函数（修复梯度问题）
def pde_residual(model, A, t, pi=100.0, beta=0.5, alpha=1.2, r=0.05, T=4.0, tau=3.0):
    # 确保输入张量需要梯度
    A = A.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)
    
    # 计算导数
    C = model(A, t)
    
    # 计算dC_dt
    dC_dt = torch.autograd.grad(
        C, t, 
        grad_outputs=torch.ones_like(C),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算dC_dA
    dC_dA = torch.autograd.grad(
        C, A, 
        grad_outputs=torch.ones_like(C),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算d2C_dA2
    d2C_dA2 = torch.autograd.grad(
        dC_dA, A, 
        grad_outputs=torch.ones_like(dC_dA),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算相关项
    z_t = beta * (T - t).pow(-alpha)
    A_pi_ratio = A / pi
    A_pi_ratio = A_pi_ratio.clamp(min=1e-6, max=1-1e-6)
    
    # 使用标准正态分布的逆CDF
    Phi_inv = torch.distributions.normal.Normal(0, 1).icdf(A_pi_ratio)
    Phi_prime = torch.exp(-0.5 * Phi_inv.pow(2)) / torch.sqrt(torch.tensor(2.0 * torch.pi))
    
    # 非线性项
    nonlinear_term = 0.5 * d2C_dA2 * (pi * Phi_prime * torch.sqrt(z_t)).pow(2)
    
    # PDE残差
    residual = dC_dt + nonlinear_term - r * C
    
    return residual


# 计算残差
n_residual = 500  # 减少点数以提高速度
residual_A = np.linspace(A_min, A_max, n_residual)
residual_t = np.linspace(t_min, t_max, n_residual)

# 创建网格
residual_A_tensor = torch.tensor(residual_A, dtype=torch.float32, device=device).view(-1, 1)
residual_t_tensor = torch.tensor(residual_t, dtype=torch.float32, device=device).view(-1, 1)

# 计算残差
residual = pde_residual(model, residual_A_tensor, residual_t_tensor).cpu().detach().numpy()

# 转换为NumPy数组
residual_A_np = residual_A_tensor.cpu().numpy().flatten()
residual_t_np = residual_t_tensor.cpu().numpy().flatten()

print("Residual calculation completed successfully")


# Residual vs Asset Price
ax7 = plt.subplot(gs_bottom[1])
ax7.scatter(residual_A_np, residual, alpha=0.6, color="blue", s=30)
ax7.axhline(0, color="black", linestyle="--", linewidth=2)
ax7.set_xlabel("Asset Price (A)")
ax7.set_ylabel("PDE Residual")
ax7.set_title("(f) Residual vs Asset Price")
ax7.grid(True, alpha=0.3)
ax7.set_ylim(-10, 10)

# Residual vs Time
ax8 = plt.subplot(gs_bottom[2])
ax8.scatter(residual_t_np, residual, alpha=0.6, color="red", s=30)
ax8.axhline(0, color="black", linestyle="--", linewidth=2)
ax8.set_xlabel("Time (t)")
ax8.set_ylabel("PDE Residual")
ax8.set_title("(g) Residual vs Time")
ax8.grid(True, alpha=0.3)
ax8.set_ylim(-10, 10)


# Adjust layout and save main figure
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
main_fig_path = os.path.join(SAVE_DIR, "three_row_comparison.png")
plt.savefig(main_fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Three-row comparison figure saved to: {main_fig_path}")

# Print summary statistics
print("\n=== Error Statistics ===")
print(f"Mean Absolute Error: {np.mean(np.abs(abs_error)):.6f}")
print(f"Maximum Absolute Error: {np.max(np.abs(abs_error)):.6f}")
print(f"Mean Relative Error: {np.mean(np.abs(rel_error)):.2f}%")
print(f"Maximum Relative Error: {np.max(np.abs(rel_error)):.2f}%")

In [ ]:
np.abs(rel_error)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import os
import glob
import re
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec

# 设置绘图样式
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})

# ================= Set Paths =================
RESULTS_ROOT = "./pinn_results"

# Find latest PINN run directory
pinn_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_pi100_K50_*")))
if not pinn_runs:
    raise FileNotFoundError("No PINN results directory found")
pinn_dir = pinn_runs[-1]  # take the latest run

# Find latest Monte Carlo run directory
mc_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_mc_pi100_K50_*")))
if not mc_runs:
    raise FileNotFoundError("No Monte Carlo results directory found")
mc_dir = mc_runs[-1]  # take the latest run

# Create comparison results directory
comparison_dir = os.path.join("generated", "comparison_results")
os.makedirs(comparison_dir, exist_ok=True)
print(f"Comparison results will be saved to: {comparison_dir}")

# Load parameters from PINN run (assume parameters are similar for both)
try:
    with open(os.path.join(pinn_dir, "simulation_parameters.txt"), "r") as f:
        params_text = f.read()
        # Extract key parameters
        tau = float(re.search(r"Tau = ([\d.]+)", params_text).group(1))
        K = float(re.search(r"Strike price \(K\) = ([\d.]+)", params_text).group(1))
        pi = float(re.search(r"Pi = ([\d.]+)", params_text).group(1))
except:
    # Default values if file not found
    tau = 3.0
    K = 50.0
    pi = 100.0

# ================= Load Data =================
# Load PINN results
pinn_data = np.load(os.path.join(pinn_dir, "pinn_results.npy"))
A_pinn = pinn_data[:, 0]
t_pinn = pinn_data[:, 1]
C_pinn = pinn_data[:, 2]

# Load Monte Carlo results
mc_data = np.load(os.path.join(mc_dir, "mc_results.npy"))
A_mc = mc_data[:, 0]
t_mc = mc_data[:, 1]
C_mc = mc_data[:, 2]

# Print basic info
print(f"PINN data points: {len(A_pinn)}")
print(f"Monte Carlo data points: {len(A_mc)}")
print(f"PINN price range: {C_pinn.min():.2f} - {C_pinn.max():.2f}")
print(f"Monte Carlo price range: {C_mc.min():.2f} - {C_mc.max():.2f}")

# ================= Data Alignment =================
# Create common grid
A_min = max(min(A_pinn), min(A_mc))
A_max = min(max(A_pinn), max(A_mc))
t_min = max(min(t_pinn), min(t_mc))
t_max = min(max(t_pinn), max(t_mc))

print(f"Common region: A={A_min:.1f}-{A_max:.1f}, t={t_min:.1f}-{t_max:.1f}")

# Create interpolation grid
grid_points = 100
A_grid = np.linspace(A_min, A_max, grid_points)
t_grid = np.linspace(t_min, t_max, grid_points)
A_mesh, t_mesh = np.meshgrid(A_grid, t_grid, indexing='ij')

# Interpolate PINN data to grid
C_pinn_grid = griddata(
    (A_pinn, t_pinn), 
    C_pinn, 
    (A_mesh, t_mesh), 
    method='cubic'
)

# Interpolate Monte Carlo data to grid
C_mc_grid = griddata(
    (A_mc, t_mc), 
    C_mc, 
    (A_mesh, t_mesh), 
    method='cubic'
)

# Calculate errors
abs_error = C_pinn_grid - C_mc_grid
rel_error = abs_error / (np.abs(C_mc_grid) + 1e-6) * 100  # percentage error

# For plotting, create flattened data
A_flat = A_mesh.ravel()
t_flat = t_mesh.ravel()
C_pinn_flat = C_pinn_grid.ravel()
C_mc_flat = C_mc_grid.ravel()
abs_error_flat = abs_error.ravel()
rel_error_flat = rel_error.ravel()

# ================= Figure 1: 3D Surface Comparison =================
print("Generating 3D surface comparison...")

fig = plt.figure(figsize=(20, 16))
fig.suptitle('Carbon Option Pricing: PINN vs Monte Carlo Simulation', fontsize=16)

# 1. PINN results
ax1 = fig.add_subplot(221, projection='3d')
surf1 = ax1.plot_surface(
    A_mesh, t_mesh, C_pinn_grid, 
    cmap='viridis', 
    rstride=1, 
    cstride=1,
    alpha=0.8,
    antialiased=True
)
ax1.set_xlabel('Asset Price (A)')
ax1.set_ylabel('Time (t)')
ax1.set_zlabel('Option Price (C)')
ax1.set_title('PINN Solution')
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, label='Option Price (C)')

# 2. Monte Carlo results
ax2 = fig.add_subplot(222, projection='3d')
surf2 = ax2.plot_surface(
    A_mesh, t_mesh, C_mc_grid, 
    cmap='viridis', 
    rstride=1, 
    cstride=1,
    alpha=0.8,
    antialiased=True
)
ax2.set_xlabel('Asset Price (A)')
ax2.set_ylabel('Time (t)')
ax2.set_zlabel('Option Price (C)')
ax2.set_title('Monte Carlo Simulation')
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, label='Option Price (C)')

# 3. Absolute error
ax3 = fig.add_subplot(223, projection='3d')
# Use diverging colormap for positive/negative errors
error_cmap = cm.coolwarm
surf3 = ax3.plot_surface(
    A_mesh, t_mesh, abs_error, 
    cmap=error_cmap, 
    rstride=1, 
    cstride=1,
    alpha=0.8,
    antialiased=True
)
ax3.set_xlabel('Asset Price (A)')
ax3.set_ylabel('Time (t)')
ax3.set_zlabel('Absolute Error')
ax3.set_title('Absolute Error (PINN - MC)')
fig.colorbar(surf3, ax=ax3, shrink=0.5, aspect=10, label='Absolute Error')

# 4. Relative error
ax4 = fig.add_subplot(224, projection='3d')
surf4 = ax4.plot_surface(
    A_mesh, t_mesh, rel_error, 
    cmap=error_cmap, 
    rstride=1, 
    cstride=1,
    alpha=0.8,
    antialiased=True
)
ax4.set_xlabel('Asset Price (A)')
ax4.set_ylabel('Time (t)')
ax4.set_zlabel('Relative Error (%)')
ax4.set_title('Relative Error (PINN - MC) / MC')
fig.colorbar(surf4, ax=ax4, shrink=0.5, aspect=10, label='Relative Error (%)')

plt.tight_layout(rect=[0, 0, 1, 0.96])
surface_comp_path = os.path.join(comparison_dir, "3d_surface_comparison.png")
plt.savefig(surface_comp_path, dpi=300, bbox_inches='tight')
plt.show()

# ================= Figure 2: Error Distribution =================
print("Generating error distribution plots...")

plt.figure(figsize=(12, 8))

# Absolute error distribution
plt.subplot(2, 2, 1)
plt.hist(abs_error_flat, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.xlabel('Absolute Error')
plt.ylabel('Frequency')
plt.title('Absolute Error Distribution')
plt.grid(True, alpha=0.3)

# Relative error distribution
plt.subplot(2, 2, 2)
plt.hist(rel_error_flat, bins=50, color='salmon', edgecolor='black', alpha=0.7)
plt.xlabel('Relative Error (%)')
plt.ylabel('Frequency')
plt.title('Relative Error Distribution')
plt.grid(True, alpha=0.3)

# Error boxplot
plt.subplot(2, 2, 3)
plt.boxplot([abs_error_flat, rel_error_flat], 
            labels=['Absolute Error', 'Relative Error(%)'], 
            patch_artist=True,
            boxprops=dict(facecolor='lightgreen'))
plt.ylabel('Error Value')
plt.title('Error Boxplot')
plt.grid(True, alpha=0.3)

# Error scatter plot
plt.subplot(2, 2, 4)
plt.scatter(C_mc_flat, abs_error_flat, 
            c=np.abs(rel_error_flat), 
            cmap='viridis', 
            alpha=0.6,
            s=10)
plt.colorbar(label='Relative Error (%)')
plt.xlabel('Monte Carlo Price')
plt.ylabel('Absolute Error')
plt.title('Price vs Error')
plt.grid(True, alpha=0.3)

plt.tight_layout()
error_dist_path = os.path.join(comparison_dir, "error_distribution.png")
plt.savefig(error_dist_path, dpi=300)
plt.show()

# ================= Figure 3: 2D Contour Comparison (Improved) =================
print("Generating improved 2D contour comparison...")

# Create symmetric colormap for errors
error_norm = colors.TwoSlopeNorm(vmin=-np.max(np.abs(abs_error)), vcenter=0, vmax=np.max(np.abs(abs_error)))

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Carbon Option Price Comparison: 2D Views', fontsize=16)

# Use GridSpec for better layout control
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1])

# First row: A-t plane with color representing C
# PINN results
ax1 = plt.subplot(gs[0, 0])
contour1 = ax1.contourf(
    A_mesh, t_mesh, C_pinn_grid, 
    levels=50, 
    cmap='viridis'
)
ax1.set_xlabel('Asset Price (A)')
ax1.set_ylabel('Time (t)')
ax1.set_title('PINN: Option Price Surface')
plt.colorbar(contour1, ax=ax1, label='Option Price (C)')

# Monte Carlo results
ax2 = plt.subplot(gs[0, 1])
contour2 = ax2.contourf(
    A_mesh, t_mesh, C_mc_grid, 
    levels=50, 
    cmap='viridis'
)
ax2.set_xlabel('Asset Price (A)')
ax2.set_ylabel('Time (t)')
ax2.set_title('Monte Carlo: Option Price Surface')
plt.colorbar(contour2, ax=ax2, label='Option Price (C)')

# Absolute error
ax3 = plt.subplot(gs[0, 2])
contour3 = ax3.contourf(
    A_mesh, t_mesh, abs_error, 
    levels=50, 
    cmap=error_cmap,
    norm=error_norm
)
ax3.set_xlabel('Asset Price (A)')
ax3.set_ylabel('Time (t)')
ax3.set_title('Absolute Error (PINN - MC)')
cbar3 = plt.colorbar(contour3, ax=ax3, label='Price Difference (C)')

# Second row: A-C plane with color representing t
# PINN results
ax4 = plt.subplot(gs[1, 0])
contour4 = ax4.contourf(
    A_mesh, C_pinn_grid, t_mesh, 
    levels=50, 
    cmap='plasma'
)
ax4.set_xlabel('Asset Price (A)')
ax4.set_ylabel('Option Price (C)')
ax4.set_title('PINN: Time Evolution')
plt.colorbar(contour4, ax=ax4, label='Time (t)')

# Monte Carlo results
ax5 = plt.subplot(gs[1, 1])
contour5 = ax5.contourf(
    A_mesh, C_mc_grid, t_mesh, 
    levels=50, 
    cmap='plasma'
)
ax5.set_xlabel('Asset Price (A)')
ax5.set_ylabel('Option Price (C)')
ax5.set_title('Monte Carlo: Time Evolution')
plt.colorbar(contour5, ax=ax5, label='Time (t)')

# Relative error
ax6 = plt.subplot(gs[1, 2])
contour6 = ax6.contourf(
    A_mesh, t_mesh, np.abs(rel_error), 
    levels=50, 
    cmap='hot'
)
ax6.set_xlabel('Asset Price (A)')
ax6.set_ylabel('Time (t)')
ax6.set_title('Absolute Relative Error (%)')
cbar6 = plt.colorbar(contour6, ax=ax6, label='Relative Error (%)')

plt.tight_layout(rect=[0, 0, 1, 0.96])
contour_comp_path = os.path.join(comparison_dir, "2d_contour_comparison.png")
plt.savefig(contour_comp_path, dpi=300, bbox_inches='tight')
plt.show()

# ================= Figure 4: Time Series Comparison =================
print("Generating time series comparison...")

# Select 6 time points evenly spaced
time_points = np.linspace(t_min, t_max, 6)
colors = cm.viridis(np.linspace(0, 1, len(time_points)))

plt.figure(figsize=(12, 8))

# For each time point, plot PINN and MC results
for i, t_val in enumerate(time_points):
    # Find the closest time index in the grid
    t_idx = np.argmin(np.abs(t_grid - t_val))
    
    # Extract results at this time
    A_at_t = A_grid
    pinn_at_t = C_pinn_grid[:, t_idx]
    mc_at_t = C_mc_grid[:, t_idx]
    
    # Plot both methods
    plt.plot(A_at_t, pinn_at_t, 
             linewidth=2.5, 
             linestyle='-', 
             label=f"PINN t={t_val:.1f}" if i == 0 else "",
             color=colors[i])
    
    plt.plot(A_at_t, mc_at_t, 
             linewidth=2, 
             linestyle='--', 
             label=f"MC t={t_val:.1f}" if i == 0 else "",
             color=colors[i])

# Add a single legend for all lines
plt.plot([], [], 'k-', linewidth=2.5, label='PINN')
plt.plot([], [], 'k--', linewidth=2, label='Monte Carlo')
plt.legend()

plt.xlabel("Asset Price (A)")
plt.ylabel("Option Price (C)")
plt.title(f"Carbon Option Price at Different Times (K={K})")
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add expiration line
plt.axvline(x=K, color='r', linestyle=':', alpha=0.7, label='Strike Price')
plt.legend(loc='upper left')

time_series_path = os.path.join(comparison_dir, "time_series_comparison.png")
plt.savefig(time_series_path, dpi=300)
plt.show()


# ================= Summary Statistics =================
# Calculate error statistics
mean_abs_error = np.mean(np.abs(abs_error_flat))
mean_rel_error = np.mean(np.abs(rel_error_flat))
max_abs_error = np.max(np.abs(abs_error_flat))
max_rel_error = np.max(np.abs(rel_error_flat))

# Save statistics
stats_info = f"""
Comparison Statistics:
----------------------
Data points: {len(A_flat)}
Common region: A={A_min:.1f} to {A_max:.1f}, t={t_min:.1f} to {t_max:.1f}

Mean Absolute Error: {mean_abs_error:.4f}
Mean Absolute Relative Error: {mean_rel_error:.2f}%
Max Absolute Error: {max_abs_error:.4f}
Max Absolute Relative Error: {max_rel_error:.2f}%

Monte Carlo Mean Price: {np.mean(C_mc_flat):.4f}
PINN Mean Price: {np.mean(C_pinn_flat):.4f}
"""

stats_path = os.path.join(comparison_dir, "comparison_statistics.txt")
with open(stats_path, "w") as f:
    f.write(stats_info)

print(stats_info)
print(f"All comparison results saved to: {comparison_dir}")

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib import cm
# from mpl_toolkits.mplot3d import Axes3D
# from scipy.interpolate import griddata
# import os
# import glob
# import re
# import matplotlib.colors as colors

# # ================= 设置路径 =================
# RESULTS_ROOT = "./pinn_results"

# # 查找最新的PINN运行目录
# pinn_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_pi100_K50_*")))
# if not pinn_runs:
#     raise FileNotFoundError("未找到PINN结果目录")
# pinn_dir = pinn_runs[-1]  # 取最新的运行

# # 查找最新的蒙特卡洛运行目录
# mc_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_mc_pi100_K50_*")))
# if not mc_runs:
#     raise FileNotFoundError("未找到蒙特卡洛结果目录")
# mc_dir = mc_runs[-1]  # 取最新的运行

# # 创建比较结果保存目录
# comparison_dir = os.path.join("generated", "comparison_results")
# os.makedirs(comparison_dir, exist_ok=True)
# print(f"比较结果将保存到: {comparison_dir}")

# # ================= 加载数据 =================
# # 加载PINN结果
# pinn_data = np.load(os.path.join(pinn_dir, "pinn_results.npy"))
# A_pinn = pinn_data[:, 0]
# t_pinn = pinn_data[:, 1]
# C_pinn = pinn_data[:, 2]

# # 加载蒙特卡洛结果
# mc_data = np.load(os.path.join(mc_dir, "mc_results.npy"))
# A_mc = mc_data[:, 0]
# t_mc = mc_data[:, 1]
# C_mc = mc_data[:, 2]

# # 打印基本信息
# print(f"PINN数据点数: {len(A_pinn)}")
# print(f"蒙特卡洛数据点数: {len(A_mc)}")
# print(f"PINN价格范围: {C_pinn.min():.2f} - {C_pinn.max():.2f}")
# print(f"蒙特卡洛价格范围: {C_mc.min():.2f} - {C_mc.max():.2f}")

# # ================= 数据对齐 =================
# # 创建共同的网格
# A_min = max(min(A_pinn), min(A_mc))
# A_max = min(max(A_pinn), max(A_mc))
# t_min = max(min(t_pinn), min(t_mc))
# t_max = min(max(t_pinn), max(t_mc))

# print(f"共同区域: A={A_min:.1f}-{A_max:.1f}, t={t_min:.1f}-{t_max:.1f}")

# # 创建插值网格
# grid_points = 100
# A_grid = np.linspace(A_min, A_max, grid_points)
# t_grid = np.linspace(t_min, t_max, grid_points)
# A_mesh, t_mesh = np.meshgrid(A_grid, t_grid, indexing='ij')

# # 将PINN数据插值到网格
# C_pinn_grid = griddata(
#     (A_pinn, t_pinn), 
#     C_pinn, 
#     (A_mesh, t_mesh), 
#     method='cubic'
# )

# # 将蒙特卡洛数据插值到网格
# C_mc_grid = griddata(
#     (A_mc, t_mc), 
#     C_mc, 
#     (A_mesh, t_mesh), 
#     method='cubic'
# )

# # 计算误差
# abs_error = C_pinn_grid - C_mc_grid
# rel_error = abs_error / (np.abs(C_mc_grid) + 1e-6) * 100  # 百分比误差

# # 为绘图准备展平的数据
# A_flat = A_mesh.ravel()
# t_flat = t_mesh.ravel()
# C_pinn_flat = C_pinn_grid.ravel()
# C_mc_flat = C_mc_grid.ravel()
# abs_error_flat = abs_error.ravel()
# rel_error_flat = rel_error.ravel()

# # ================= 图1: 3D曲面比较 =================
# print("生成3D曲面比较图...")

# fig = plt.figure(figsize=(20, 16))
# fig.suptitle('碳期权价格: PINN vs 蒙特卡洛模拟', fontsize=16)

# # 1. PINN结果
# ax1 = fig.add_subplot(221, projection='3d')
# surf1 = ax1.plot_surface(
#     A_mesh, t_mesh, C_pinn_grid, 
#     cmap='viridis', 
#     rstride=1, 
#     cstride=1,
#     alpha=0.8,
#     antialiased=True
# )
# ax1.set_xlabel('资产价格 (A)')
# ax1.set_ylabel('时间 (t)')
# ax1.set_zlabel('期权价格 (C)')
# ax1.set_title('PINN结果')
# fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, label='期权价格')

# # 2. 蒙特卡洛结果
# ax2 = fig.add_subplot(222, projection='3d')
# surf2 = ax2.plot_surface(
#     A_mesh, t_mesh, C_mc_grid, 
#     cmap='viridis', 
#     rstride=1, 
#     cstride=1,
#     alpha=0.8,
#     antialiased=True
# )
# ax2.set_xlabel('资产价格 (A)')
# ax2.set_ylabel('时间 (t)')
# ax2.set_zlabel('期权价格 (C)')
# ax2.set_title('蒙特卡洛模拟')
# fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, label='期权价格')

# # 3. 绝对误差
# ax3 = fig.add_subplot(223, projection='3d')
# # 使用发散色图显示正负误差
# error_cmap = cm.coolwarm
# surf3 = ax3.plot_surface(
#     A_mesh, t_mesh, abs_error, 
#     cmap=error_cmap, 
#     rstride=1, 
#     cstride=1,
#     alpha=0.8,
#     antialiased=True
# )
# ax3.set_xlabel('资产价格 (A)')
# ax3.set_ylabel('时间 (t)')
# ax3.set_zlabel('绝对误差')
# ax3.set_title('绝对误差 (PINN - MC)')
# fig.colorbar(surf3, ax=ax3, shrink=0.5, aspect=10, label='绝对误差')

# # 4. 相对误差
# ax4 = fig.add_subplot(224, projection='3d')
# surf4 = ax4.plot_surface(
#     A_mesh, t_mesh, rel_error, 
#     cmap=error_cmap, 
#     rstride=1, 
#     cstride=1,
#     alpha=0.8,
#     antialiased=True
# )
# ax4.set_xlabel('资产价格 (A)')
# ax4.set_ylabel('时间 (t)')
# ax4.set_zlabel('相对误差 (%)')
# ax4.set_title('相对误差 (PINN - MC) / MC')
# fig.colorbar(surf4, ax=ax4, shrink=0.5, aspect=10, label='相对误差 (%)')

# plt.tight_layout(rect=[0, 0, 1, 0.96])
# surface_comp_path = os.path.join(comparison_dir, "3d_surface_comparison.png")
# plt.savefig(surface_comp_path, dpi=300, bbox_inches='tight')
# plt.show()

# # ================= 图2: 误差分布图 =================
# print("生成误差分布图...")

# plt.figure(figsize=(12, 8))

# # 绝对误差分布
# plt.subplot(2, 2, 1)
# plt.hist(abs_error_flat, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
# plt.xlabel('绝对误差')
# plt.ylabel('频率')
# plt.title('绝对误差分布')
# plt.grid(True, alpha=0.3)

# # 相对误差分布
# plt.subplot(2, 2, 2)
# plt.hist(rel_error_flat, bins=50, color='salmon', edgecolor='black', alpha=0.7)
# plt.xlabel('相对误差 (%)')
# plt.ylabel('频率')
# plt.title('相对误差分布')
# plt.grid(True, alpha=0.3)

# # 误差箱线图
# plt.subplot(2, 2, 3)
# plt.boxplot([abs_error_flat, rel_error_flat], 
#             labels=['绝对误差', '相对误差(%)'], 
#             patch_artist=True,
#             boxprops=dict(facecolor='lightgreen'))
# plt.ylabel('误差值')
# plt.title('误差箱线图')
# plt.grid(True, alpha=0.3)

# # 误差散点图
# plt.subplot(2, 2, 4)
# plt.scatter(C_mc_flat, abs_error_flat, 
#             c=np.abs(rel_error_flat), 
#             cmap='viridis', 
#             alpha=0.6,
#             s=10)
# plt.colorbar(label='相对误差 (%)')
# plt.xlabel('蒙特卡洛价格')
# plt.ylabel('绝对误差')
# plt.title('价格 vs 误差')
# plt.grid(True, alpha=0.3)

# plt.tight_layout()
# error_dist_path = os.path.join(comparison_dir, "error_distribution.png")
# plt.savefig(error_dist_path, dpi=300)
# plt.show()

# # ================= 图3: 二维Contour图比较 =================
# print("生成二维Contour图比较...")

# # 创建对称的误差色图
# error_norm = colors.TwoSlopeNorm(vmin=rel_error.min(), vcenter=0, vmax=rel_error.max())

# fig, axes = plt.subplots(2, 3, figsize=(24, 16))
# fig.suptitle('碳期权价格二维视图比较', fontsize=20)

# # 第一行: x=A, y=C, 颜色表示t
# # PINN结果
# contour1 = axes[0, 0].contourf(
#     A_mesh, C_pinn_grid, t_mesh, 
#     levels=50, 
#     cmap='viridis'
# )
# axes[0, 0].set_xlabel('资产价格 (A)')
# axes[0, 0].set_ylabel('期权价格 (C)')
# axes[0, 0].set_title('PINN: A-C平面 (颜色表示时间)')
# fig.colorbar(contour1, ax=axes[0, 0], label='时间 (t)')

# # 蒙特卡洛结果
# contour2 = axes[0, 1].contourf(
#     A_mesh, C_mc_grid, t_mesh, 
#     levels=50, 
#     cmap='viridis'
# )
# axes[0, 1].set_xlabel('资产价格 (A)')
# axes[0, 1].set_ylabel('期权价格 (C)')
# axes[0, 1].set_title('蒙特卡洛: A-C平面 (颜色表示时间)')
# fig.colorbar(contour2, ax=axes[0, 1], label='时间 (t)')

# # 时间差异
# contour3 = axes[0, 2].contourf(
#     A_mesh, C_pinn_grid, t_mesh - t_mesh,  # 实际差异在颜色上
#     levels=50, 
#     cmap='viridis'
# )
# axes[0, 2].set_xlabel('资产价格 (A)')
# axes[0, 2].set_ylabel('期权价格 (C)')
# axes[0, 2].set_title('A-C平面 (时间无差异)')
# fig.colorbar(contour3, ax=axes[0, 2], label='时间差异 (t)')

# # 第二行: x=A, y=t, 颜色表示C
# # PINN结果
# contour4 = axes[1, 0].contourf(
#     A_mesh, t_mesh, C_pinn_grid, 
#     levels=50, 
#     cmap='viridis'
# )
# axes[1, 0].set_xlabel('资产价格 (A)')
# axes[1, 0].set_ylabel('时间 (t)')
# axes[1, 0].set_title('PINN: A-t平面 (颜色表示期权价格)')
# fig.colorbar(contour4, ax=axes[1, 0], label='期权价格 (C)')

# # 蒙特卡洛结果
# contour5 = axes[1, 1].contourf(
#     A_mesh, t_mesh, C_mc_grid, 
#     levels=50, 
#     cmap='viridis'
# )
# axes[1, 1].set_xlabel('资产价格 (A)')
# axes[1, 1].set_ylabel('时间 (t)')
# axes[1, 1].set_title('蒙特卡洛: A-t平面 (颜色表示期权价格)')
# fig.colorbar(contour5, ax=axes[1, 1], label='期权价格 (C)')

# # 价格差异
# contour6 = axes[1, 2].contourf(
#     A_mesh, t_mesh, abs_error, 
#     levels=50, 
#     cmap=error_cmap,
#     norm=error_norm
# )
# axes[1, 2].set_xlabel('资产价格 (A)')
# axes[1, 2].set_ylabel('时间 (t)')
# axes[1, 2].set_title('价格差异: PINN - MC')
# cbar = fig.colorbar(contour6, ax=axes[1, 2], label='价格差异 (C)')

# plt.tight_layout(rect=[0, 0, 1, 0.96])
# contour_comp_path = os.path.join(comparison_dir, "2d_contour_comparison.png")
# plt.savefig(contour_comp_path, dpi=300, bbox_inches='tight')
# plt.show()

# # ================= 总结统计 =================
# # 计算误差统计
# mean_abs_error = np.mean(np.abs(abs_error_flat))
# mean_rel_error = np.mean(np.abs(rel_error_flat))
# max_abs_error = np.max(np.abs(abs_error_flat))
# max_rel_error = np.max(np.abs(rel_error_flat))

# # 保存统计信息
# stats_info = f"""
# 比较结果统计:
# --------------
# 数据点数量: {len(A_flat)}
# 共同区域: A={A_min:.1f} 到 {A_max:.1f}, t={t_min:.1f} 到 {t_max:.1f}

# 平均绝对误差: {mean_abs_error:.4f}
# 平均相对误差: {mean_rel_error:.2f}%
# 最大绝对误差: {max_abs_error:.4f}
# 最大相对误差: {max_rel_error:.2f}%

# 蒙特卡洛平均价格: {np.mean(C_mc_flat):.4f}
# PINN平均价格: {np.mean(C_pinn_flat):.4f}
# """

# stats_path = os.path.join(comparison_dir, "comparison_statistics.txt")
# with open(stats_path, "w") as f:
#     f.write(stats_info)

# print(stats_info)
# print(f"所有比较结果已保存到: {comparison_dir}")